In [1]:
# Let's add the root directory to our system search path to allow imports from sibling directories.
import os, sys
module_path = os.path.abspath(os.path.join(".."))
if module_path not in sys.path:
    sys.path.append(module_path)

In [2]:
import gc, random
from pathlib import Path
from typing import List

import cv2
import torch
import numpy as np

from tqdm.notebook import tqdm

from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import SGDClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import classification_report, confusion_matrix

from src.approx_knn import ApproxKNeighborsClassifier
from src.utils import load_backbone, run_inference

WEIGHTS_DIR = Path("../weights/")
EPOCH_COUNT = len(list(WEIGHTS_DIR.iterdir())) - 2

DEVICE = torch.device("cuda")  # ConvNeXtV2 doesn't support CPU

SEED = 42
RNG = np.random.default_rng(seed=SEED)
random.seed(SEED)

DATASET_PATH = Path("../data/labelled/s3seg")
IMGS_PATH = DATASET_PATH / "images/"
MASK_PATH = DATASET_PATH / "masks/"
EMBEDS_PATH = DATASET_PATH / "embeds/"

# Few Shot Scalability (1%, 5%, 10%, 50%, 100%)
FEW_SHOT = (0.01, 0.05, 0.10, 0.50, 1.00)

NUM_SAMPLES = 10_000

DATA_EXT = '.tiff'
DATA_FILES = random.sample(sorted([im.stem for im in IMGS_PATH.glob(f"*{DATA_EXT}")]), k=NUM_SAMPLES)

N = len(DATA_FILES)
TEST_SPLIT = int(0.2 * N)

/home/hayat/projects/benthic/venv/lib/python3.8/site-packages/spconv/pytorch/functional.py:47: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  _TORCH_CUSTOM_FWD = amp.custom_fwd(cast_inputs=torch.float16)
/home/hayat/projects/benthic/venv/lib/python3.8/site-packages/spconv/pytorch/functional.py:97: FutureWarning: `torch.cuda.amp.custom_bwd(args...)` is deprecated. Please use `torch.amp.custom_bwd(args..., device_type='cuda')` instead.
  def backward(ctx, grad_output):
/home/hayat/projects/benthic/venv/lib/python3.8/site-packages/spconv/pytorch/functional.py:163: FutureWarning: `torch.cuda.amp.custom_bwd(args...)` is deprecated. Please use `torch.amp.custom_bwd(args..., device_type='cuda')` instead.
  def backward(ctx, grad_output):
/home/hayat/projects/benthic/venv/lib/python3.8/site-packages/spconv/pytorch/functional.py:243: FutureWarning: `torch.cuda.amp.custom_bwd(args...)` is deprecated. Pl

In [ ]:
def print_iou(y_true, y_pred, labels):
    cm = confusion_matrix(y_true, y_pred, labels=labels)

    print("Confusion Matrix:")

    # Create the header row for predictions
    header = f"{'True \\ Pred':>12} | " + " ".join([f"{str(label):>6}" for label in labels])
    print(header)
    print("-" * len(header))

    # Create each row with the true label and its values
    for i, row_label in enumerate(labels):
        row_str = " ".join([f"{str(val):>6}" for val in cm[i]])
        print(f"{str(row_label):>12} | {row_str}")

    print("-" * len(header))

    # --- Calculate and Print IoU ---
    intersection = np.diag(cm)
    union = cm.sum(axis=1) + cm.sum(axis=0) - intersection

    # Avoid division by zero
    iou = np.divide(intersection, union, out=np.zeros_like(intersection, dtype=float), where=union!=0)

    print("\nIoU per class:")
    for label, val in zip(labels, iou):
        print(f"  Class {str(label):<5}: {val:.3f}")

    print("-" * 20)
    print(f"Mean IoU (mIoU) : {np.mean(iou):.3f}")

In [4]:
EMBEDS_PATH.mkdir(exist_ok=True)

embed_files = list(EMBEDS_PATH.glob('*.npz'))
if len(embed_files) < N:
    model = load_backbone(WEIGHTS_DIR / "checkpoint_latest.pth")

    for file in DATA_FILES:
        embed_path = EMBEDS_PATH / (file + '.npz')
        if embed_path.exists():
            continue

        img_path = IMGS_PATH / (file + DATA_EXT)
        img = cv2.imread(str(img_path), cv2.IMREAD_UNCHANGED).astype(np.float32)
        img = torch.from_numpy(img[np.newaxis, :, :])  # (W, H) -> (1, W, H)

        cls, flat_patches, outputs = run_inference(model, img)
        np.savez_compressed(
            embed_path,
            cls=cls,
            stage0=outputs[0],
            stage1=outputs[1],
            stage2=outputs[2],
            stage3=outputs[3],
            stage4=outputs[4],  # Fused Hypercolumn
        )

In [5]:
masks, embeds = None, None
for i, file in enumerate(tqdm(DATA_FILES)):
    mask_path = MASK_PATH / (file + DATA_EXT)
    embed_path = EMBEDS_PATH / (file + '.npz')

    mask = cv2.imread(str(mask_path), cv2.IMREAD_UNCHANGED).astype(np.float32)
    data = np.load(embed_path)

    if masks is None or embeds is None:
        masks = np.zeros([N, ] + list(mask.shape), dtype=np.float32)

        embed = (data['cls'], data['stage0'], data['stage1'], data['stage2'], data['stage3'], data['stage4'])
        embeds = [
            np.zeros([N, ] + list(e.shape), dtype=np.float32)
            for e in embed
        ]

    masks[i] = mask

    embeds[0][i] = data['cls']
    embeds[1][i] = data['stage0']
    embeds[2][i] = data['stage1']
    embeds[3][i] = data['stage2']
    embeds[4][i] = data['stage3']
    embeds[5][i] = data['stage4']

    data.close()
    del mask, data

unique_labels = np.unique(masks)

  0%|          | 0/10000 [00:00<?, ?it/s]

In [6]:
probes = [[] for _ in range(5)]
for stage in range(5):
    X_train = embeds[stage + 1][TEST_SPLIT:]

    _, H_orig, W_orig = masks.shape
    _, H_feat, W_feat, C = X_train.shape

    # Exact Nearest-Neighbor mask downsampling (center-aligned)
    row_idx = ((np.arange(H_feat) + 0.5) * (H_orig / H_feat)).astype(int)
    col_idx = ((np.arange(W_feat) + 0.5) * (W_orig / W_feat)).astype(int)

    y_train = masks[TEST_SPLIT:][:, row_idx[:, None], col_idx]

    X_test = embeds[stage + 1][:TEST_SPLIT].reshape(-1, C)  # (N * H * W, C)
    y_test = masks[:TEST_SPLIT][:, row_idx[:, None], col_idx].reshape(-1)  # (N * H * W)

    for shot in FEW_SHOT:
        num_samples = int(shot * (N - TEST_SPLIT))

        X_samples = X_train[:num_samples].reshape(-1, C)  # (N * H * W, C)
        y_samples = y_train[:num_samples].reshape(-1)  # (N * H * W)

        linear_probe = make_pipeline(
            StandardScaler(),
            SGDClassifier(
                loss="log_loss",          # Logistic regression loss
                penalty="l2",             # L2 (Ridge) regularization
                alpha=1e-4,
                max_iter=1000,
                tol=1e-3,
                early_stopping=True,
                validation_fraction=0.1,
                class_weight="balanced",
                random_state=SEED,
                n_jobs=-1
            )
        )

        linear_probe.fit(X_samples, y_samples)
        lin_train_preds = linear_probe.predict(X_samples)
        lin_test_preds = linear_probe.predict(X_test)

        print(f"=== Stage {stage + 1} | Few-Shot Scale: {shot * 100:.0f}% ===")

        print("\n-- Linear Probe --")
        print("Train Set:")
        print(classification_report(y_samples, lin_train_preds, digits=3))
        print_iou(y_samples, lin_train_preds, unique_labels)

        print("\nTest Set:")
        print(classification_report(y_test, lin_test_preds, digits=3))
        print_iou(y_test, lin_test_preds, unique_labels)

        knn_probe = make_pipeline(
            StandardScaler(),
            ApproxKNeighborsClassifier(
                n_neighbors=10,  # Based on Two-NN test
                weights="distance",
            )
        )

        knn_probe.fit(X_samples, y_samples)
        knn_train_preds = knn_probe.predict(X_samples)
        knn_test_preds = knn_probe.predict(X_test)

        print("\n-- K-Nearest Neighbors --")
        print("Train Set:")
        print(classification_report(y_samples, knn_train_preds, digits=3))
        print_iou(y_samples, knn_train_preds, unique_labels)

        print("\nTest Set:")
        print(classification_report(y_test, knn_test_preds, digits=3))
        print_iou(y_test, knn_test_preds, unique_labels)

        print("\n" + "="*50 + "\n\n")

        probes[stage].append(linear_probe)  # Only save the linear probe

        del knn_probe, knn_train_preds, knn_test_preds, lin_train_preds, lin_test_preds

    del X_train, X_test
    del y_train, y_test


=== Stage 1 | Few-Shot Scale: 1% ===

-- Linear Probe --
Train Set:
              precision    recall  f1-score   support

         0.0      0.828     0.831     0.829    157503
         1.0      0.682     0.682     0.682     94427
         2.0      0.345     0.319     0.332     35337
         3.0      0.660     0.694     0.677     40413

    accuracy                          0.716    327680
   macro avg      0.629     0.631     0.630    327680
weighted avg      0.713     0.716     0.714    327680

IoU per class:
  Class 0.0: 0.709
  Class 1.0: 0.517
  Class 2.0: 0.199
  Class 3.0: 0.511
Mean IoU (mIoU): 0.484

Test Set:
              precision    recall  f1-score   support

         0.0      0.805     0.836     0.820   4224912
         1.0      0.648     0.662     0.655   1983575
         2.0      0.263     0.249     0.256   1034718
         3.0      0.526     0.439     0.479    948795

    accuracy                          0.674   8192000
   macro avg      0.560     0.547     0.552   

KeyboardInterrupt: 